In [22]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1NOaP6AZomaglEAqMbDJilwZWBtSc58izZj_MC_P6Kzk"
SHEET_NAME = "Feuille 1"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()

import duckdb

# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [24]:

duck.sql(
    f"""
    create or replace table amt as 
    SELECT *
    FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true
    )
"""
)

In [25]:
duck.sql(
    """
    select * from amt 
    """
)

┌────────────┬─────────────────────────────────────┬────────────┬────────────────┬────────────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────┬────────────────┬───────────┬────────────────────────┬────────────┬──────────┬────────────────┬────────────────────┬──────────────────┬────────────────────┐
│   Datum    │                Kunde                │ ID-Nummer  │ Erbringungsart │      Leistung      │ Leistungsbeschreibung │                                        Erläuterung                                        │ Dauer Leistung │ Fahrtzeit │ Vor- und Nachbereitung │ Gesamtzeit │ Standort │   Kategorie    │     Beleister      │ Zeiterfassung GB │ Zeiterfassung Spez │
│  varchar   │               varchar               │  varchar   │    varchar     │      varchar       │        varchar        │                                          varchar                                          │    varchar     │  varchar  │

In [17]:
duck.sql(
    """
    with cleaned as (
        select * replace(replace(Gesamtzeit, ',', '.')::float as Gesamtzeit) 
        from amt 
        where Kunde is not null and Leistung = 'Grundbetreuung'
    )
    select "ID-Nummer", Kategorie, sum(Gesamtzeit)
    from cleaned 
    group by 1, 2
    """
).to_csv('basic_care_amt.csv')

In [29]:
duck.sql(
    """
    create or replace table pg.bas_firms.basic_care_hours_done as
    with cleaned as (
        select * replace(
            replace(Gesamtzeit, ',', '.')::float as Gesamtzeit,
            strptime(Datum, '%d/%m/%Y')::date as Datum
        )
        from amt 
        where Kunde is not null and Leistung = 'Grundbetreuung'
    )
    select * 
    from cleaned
    """
)

In [ ]:
duck.sql(
    """
    select "ID-Nummer", 
    from amt 
    where Kunde is not null

    """
)